> ### Note on Labs and Assigments:
>
> 🔧 Look for the **wrench emoji** 🔧 — it highlights where you're expected to take action!
>
> These sections are graded and are not optional.
>

# **IS 4487 LAB 6 - DATA CLEANING**

## Outline

- Load and inspect a new dataset (Megatelco)
- Fix column names: reformatting string variables
- Fix data types
- Handle missing values
- Remove duplicate rows
- Review and remove outliers
- Reflect on data quality

In this lab, we’ll clean the data to get it ready for transformations and analysis.

We will continue working with this dataset in **Lab 7**, where we will create new features and apply transformations.

<a href="https://colab.research.google.com/github/vandanara/UofUtah_IS4487/blob/main/Labs/lab_06_data_cleaning.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Business Context: Churn at a telecom company

MegaTelCo is one of the largest telecommunication firms in the United States. They are having a major problem with customer retention in their wireless business. In the mid-Atlantic region, 20% of cell phone customers leave when their contracts expire, and it is getting increasingly difficult to acquire new customers. Since the cell phone market is now saturated, the huge growth in the wireless market has tapered off.

Communications companies are often engaged in battles to attract each other's customers while retaining their own. Customers leaving for competitors is labeled **churn**. Customer churn is a major business problem for telecom companies; leads to lost revenue for one company and higher costs for another to acquire new customers. Telecom providers want to understand why customers leave so they can take action—such as improving service, offering promotions, or adjusting plans—to retain them.

We have been called in to help understand the problem and to devise a solution.

Source: Provost, Foster, Fawcett, Tom. (2013). Data Science for Business: What You Need to Know about Data Mining and Data-Analytic Thinking . Sebastopol, California: O\'Reilly.

## Megatelco Data Dictionary

Megatelco dataset provides a rich set of variables that can help analyze and predict churn by combining different perspectives on the customer. Demographic variables (like income and home value) can reveal differences in customer segments, while usage variables (such as data overages, call behavior, and texting) help identify whether a customer’s plan fits their actual usage. Phone-related variables (like operating system and handset price) may indicate customer preferences or investment in the service, and attitudinal variables (like satisfaction and intent to switch) give direct insight into customer sentiment. By using the `Leave` variable as the outcome/target, analysts can look for patterns—such as whether dissatisfied customers with frequent overages are more likely to churn—and build models to predict which current customers are at risk. This allows companies to proactively target those customers with retention strategies, making churn analysis both a practical and highly valuable application of data analytics

 DEMOGRAPHIC VARIABLES:
 - College - has the customer attended some college (one, zero)
 - Income - annual income of customer
 - House - estimated price of the customer's home (if applicable)

 USAGE VARIABLES:
 - Data Overage Mb - Average number of megabytes that the customer used in excess of the plan limit (over last 12 months)
 - Data Leftover Mb - Average number of megabytes that the customer use was below the plan limit (over last 12 months)
 - Data Mb Used - Average number of megabytes used per month (over last 12 months)
 - Text Message Count - Average number of texts per month (over last 12 months)
 - Over 15 Minute Calls Per Month - Average number of calls over 15 minutes in duration per month (over last 12 months)
 - Average Call Duration- Average call duration (over last 12 months)

PHONE VARIABLES:
 - Operating System - Current operating system of phone (IOS, Android)
 - Handset Price - Retail price of the phone used by the customer

ATTITUDINAL VARIABLES:
 - Reported Satisfaction - Survey response to "How satisfied are you with your current phone plan?" (high, avg, low)
 - Reported Usage Level - Survey response to "How much do your use your phone?" (high, avg, low)
 - Considering Change of Plan - Survey response to "Are you currently planning to change companies when your contract expires?" (yes, no)

OTHER VARIABLES
 - Leave - Did this customer churn with the last contract expiration? (LEAVE, STAY)
 - ID - numeric Customer identifier

In [64]:
import pandas as pd

url = "https://raw.githubusercontent.com/vandanara/UofUtah_IS4487/refs/heads/main/DataSets/megatelco_leave_survey_data_cleaning.csv"
df = pd.read_csv(url)

df.head()

,college,income,data_overage_mb,data_leftover_mb,data_mb_used,text_message_count,house,handset_price,over_15mins_calls_per_month,average_call_duration,reported_satisfaction,reported_usage_level,considering_change_of_plan,leave,id,operating_system
0,one,403137.0,70,0.0,6605.0,199,841317,653.0,5.0,8.0,low,low,yes,LEAVE,8183,Android
1,zero,129700.0,67,16.0,6028.0,134,476664,1193.0,5.0,5.0,low,low,yes,LEAVE,12501,IOS
2,zero,69741.0,60,0.0,1482.0,176,810225,1037.0,3.0,8.0,low,low,yes,STAY,7425,IOS
3,one,377572.0,0,22.0,3005.0,184,826967,1161.0,0.0,5.0,low,low,no,LEAVE,13488,IOS
4,zero,382080.0,0,0.0,1794.0,74,951896,1023.0,0.0,14.0,low,low,yes,STAY,11389,IOS


## **Part 1: Review Column Names and Structure**

Before cleaning, check the structure of the dataset:

- Check the shape
- Are column names good?
    - consistent (lowercase, no spaces)?
    - any typos or redundant labels?


Why this matters:
Inconsistent or messy column names can break code and make analysis harder to follow.




In [65]:
# rows and columns
print("The shape is: ", df.shape)

# Get column info and data types
print(df.info())

The shape is:  (15016, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15016 entries, 0 to 15015
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   college                      15016 non-null  object 
 1   income                       15006 non-null  float64
 2   data_overage_mb              15016 non-null  int64  
 3   data_leftover_mb             14916 non-null  float64
 4   data_mb_used                 14916 non-null  float64
 5   text_message_count           15016 non-null  int64  
 6   house                        15016 non-null  int64  
 7   handset_price                14916 non-null  float64
 8   over_15mins_calls_per_month  15013 non-null  float64
 9   average_call_duration        14916 non-null  float64
 10  reported_satisfaction        15016 non-null  object 
 11  reported_usage_level         15016 non-null  object 
 12  considering_change_of_plan   14201 non-null  ob

Let us standardize the column names by replacing whitespaces with an underscore

In [66]:
# Standardize column names: make lowercase, with no spaces
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

### Re-order columns

Sometimes we may wish to reorder some columns to make for easier viewing. Let us move the column `id` to the front.

In [67]:
# Get a list of all columns
cols = df.columns.tolist()

# Move 'id' to the front
cols.insert(0, cols.pop(cols.index('id')))

# Reindex the DataFrame with the new column order
df = df[cols]


### Create a deep copy

Sometimes, we may want to create a copy of our dataframe in its current state before we apply any further changes for later use.

In [68]:
# create a copy of your dataset for use in part 4
copied_df = df.copy(deep=True)

### View descriptive / summary stats of numeric and date type columns

In [69]:
# get min, max, mean, std dev of numeric and date type variables
display(df.describe())


,id,income,data_overage_mb,data_leftover_mb,data_mb_used,text_message_count,house,handset_price,over_15mins_calls_per_month,average_call_duration
count,15016.000000,15006.000000,15016.000000,14916.000000,14916.000000,15016.000000,1.501600e+04,14916.000000,15013.00000,14916.000000
mean,11856.541289,242013.863455,153.430674,37.487664,4200.979686,135.946590,8.771293e+05,794.937249,10.56551,10.060941
std,6812.183367,109627.859666,113.019892,28.052318,2203.802446,62.934783,2.870168e+05,1238.997927,8.40421,41.188957
min,2.000000,-65000.000000,0.000000,0.000000,400.000000,52.000000,-4.630000e+02,-200.000000,0.00000,1.000000
25%,6135.000000,147818.500000,54.000000,12.000000,2292.750000,93.000000,6.444678e+05,498.000000,3.00000,5.000000
50%,11754.500000,241750.500000,151.000000,34.000000,4220.000000,135.000000,8.762530e+05,777.000000,9.00000,10.000000
75%,17390.500000,336442.000000,242.000000,62.000000,6079.250000,178.000000,1.098829e+06,1063.000000,17.00000,14.000000
max,25354.000000,432000.000000,380.000000,89.000000,8000.000000,5000.000000,1.456389e+06,125000.000000,35.00000,5000.000000


### View counts of categorical column values

Note that `df.describe()` only provides summary for numeric and date type  variables. For variables defined as ***object*** - which are *text*, some maybe ***category*** (with limited and fixed number of allowed values), and others may be true ***string*** (can be any text, not limited in value).

If we suspect that a variable defined as object is potentially categorical, we will want to know what values are included. We can do this using `df[colname].value_counts()`

In [70]:
display(df['college'].value_counts())
display(df['reported_satisfaction'].value_counts())
display(df['reported_usage_level'].value_counts())
display(df['considering_change_of_plan'].value_counts())
display(df['operating_system'].value_counts())
display(df['leave'].value_counts())

,count
college,
zero,7960
one,7056


,count
reported_satisfaction,
low,10850
high,3415
avg,751


,count
reported_usage_level,
low,12235
high,2536
avg,245


,count
considering_change_of_plan,
yes,9267
no,4934


,count
operating_system,
Android,7813
IOS,7203


,count
leave,
STAY,7532
LEAVE,7484


## **Part 2: Convert Data Types**

Before analysis, make sure each column is stored in the correct data type. This helps avoid calculation errors, makes plotting smoother, and ensures models interpret the data correctly.

Think about:

- Are numbers accidentally stored as strings?
    - Anything defined as ***object*** is treated as string/text

- If the columns defined as object only allows or has only a few limited values possible, it should be converted to ***category***
    - is the categorical variable ***nominal*** or ***ordinal***?

- Are "yes"/"no" columns better represented as ***category*** types or ***binary*** (0/1)?
    - generally, when performing descriptive statistics and EDA, we want to define text columns with limited values as ***category*** data type.
    
Fixing data types now saves time and avoids issues later in your workflow.




In [71]:
# Check original data types
print("Original dtypes:\n", df.dtypes)


Original dtypes:
 id                               int64
college                         object
income                         float64
data_overage_mb                  int64
data_leftover_mb               float64
data_mb_used                   float64
text_message_count               int64
house                            int64
handset_price                  float64
over_15mins_calls_per_month    float64
average_call_duration          float64
reported_satisfaction           object
reported_usage_level            object
considering_change_of_plan      object
leave                           object
operating_system                object
dtype: object


In [73]:
# Convert object to nomimal categorical
# can use df[colname].astype()
obj_to_nomcat_cols = ['considering_change_of_plan', 'college', 'operating_system']
for acol in obj_to_nomcat_cols:
    df[acol] = df[acol].astype('category')

# Convert object to ordinal categorical (values have an order - list values in the order that matters)
df['reported_satisfaction'] = pd.Categorical(df['reported_satisfaction'], categories = ['low', 'avg', 'high'], ordered = True)

# Check updated data types
print("\nUpdated dtypes:\n", df.dtypes)



Updated dtypes:
 id                                int64
college                        category
income                          float64
data_overage_mb                   int64
data_leftover_mb                float64
data_mb_used                    float64
text_message_count                int64
house                             int64
handset_price                   float64
over_15mins_calls_per_month     float64
average_call_duration           float64
reported_satisfaction          category
reported_usage_level             object
considering_change_of_plan     category
leave                            object
operating_system               category
dtype: object


### **🔧 Try It Yourself – Part 2**

2.1. Convert the `leave` column from "STAY"/"LEAVE" to a **nominal categorical** type

2.2. Convert `reported_usage_level` to an **ordered/ordinal categorical** type

2.3. Use `.info()` to confirm the changes


In [75]:
# 🔧 2.1. add code here


# 🔧 2.2. add code here


# 🔧 2.3. add code here



## **Part 3: Handle Missing Values**

Missing data can break charts, skew stats, and disrupt models — so it needs to be handled carefully.

### Think about:
- Are the missing values random or patterned?
- Can we drop rows, or do we need to fill them?
- Should we use mean, median, or something else?

### Guidelines:
- **Drop rows** only if few are missing and the column is essential.
- Use **median** to replace missing values in numeric columns with outliers.
- Use **0** to replace missing values if missing logically means "none".
- Use **mode** to replace missing categorical values.

Cleaning missing values early avoids bigger problems later.

-----


### A Note on `.loc` and Warnings

When assigning/replacing/filling values to a DataFrame, especially after filtering or copying, it's best to use `.loc` to avoid **`SettingWithCopyWarning`**. This ensures that you're updating the original dataframe `df` and not a temporary view or copy of it.

Syntax:

`df.loc [rows to select, cols to select]`\
`df.loc[:, colname]` --> : means select all rows for given colname


In [77]:
# View missing value counts
print("Missing values per column:\n", df.isnull().sum())

# Fill 'handset_price' with median
df['handset_price'] = df['handset_price'].fillna(df['handset_price'].median())

# Drop rows with missing 'income' (if very few)
df = df.dropna(subset=['income']).copy()

# Fill missing 'data_leftover_mb' with 0 if it logically means no leftover data
df.loc[:, 'data_leftover_mb'] = df['data_leftover_mb'].fillna(0)

# Fill 'average_call_duration' with median if necessary
df.loc[:, 'average_call_duration'] = df['average_call_duration'].fillna(df['average_call_duration'].median())

# Fill 'data_mb_used' with median
df.loc[:, 'data_mb_used'] = df['data_mb_used'].fillna(df['data_mb_used'].median())

# Confirm updated missing values
print("\nMissing values after handling:\n", df.isnull().sum())


Missing values per column:
 id                               0
college                          0
income                          10
data_overage_mb                  0
data_leftover_mb               100
data_mb_used                   100
text_message_count               0
house                            0
handset_price                  100
over_15mins_calls_per_month      3
average_call_duration          100
reported_satisfaction            0
reported_usage_level             0
considering_change_of_plan     815
leave                            0
operating_system                 0
dtype: int64

Missing values after handling:
 id                               0
college                          0
income                           0
data_overage_mb                  0
data_leftover_mb                 0
data_mb_used                     0
text_message_count               0
house                            0
handset_price                    0
over_15mins_calls_per_month      3
average_call_dur

### 🔧 **Try It Yourself - Part 3**


There are still some missing values in:

- `over_15mins_calls_per_month`
- `considering_change_of_plan`

Decide how to handle them based on what makes the most sense:

- Should you fill them with 0, the median, or something else?
- For categories, would a placeholder like "unknown" or the most common value work?
  - if you use `mode()`, keep in mind it returns a Series (multiple values) since it is possible to have more than one equally most frequent value, write `mode()[0]` to pick the first one
  - if instead you wish to add a new categorical value, 'unknown' you have to first set it as one of the allowed values, and then use 'unknown' as a fill value.
- Or is it better to drop those rows?

3.1. - 3.2. Write and execute code to handle the missing values in the above remaining columns mentioned above.

3.3. Use `df.isnull().sum()` to confirm all missing values are handled.



In [51]:
# 🔧 3.1. Add code here


# 🔧 3.2. Add code here


# 🔧 3.3. Add code here

## **Part 4: Remove Duplicate Rows**

Sometimes the same row appears more than once due to data entry or processing mistakes. It's important to check for and remove these duplicates.

Think about:
- Are there rows that are exactly the same?
- If duplicates exist, should you keep the first one, the last one, or none?

Why this matters:
Duplicate rows can inflate totals, distort statistics, and lead to inaccurate conclusions (due to extra weight given to those observations incorrectly).


In [52]:
# Check for exact duplicates
print(f"Number of duplicate rows: {df.duplicated().sum()}")

# Remove them, keeping the first occurrence
df = df.drop_duplicates()

# Confirm result
print(f"Remaining rows after removing duplicates: {len(df)}")

Number of duplicate rows: 17
Remaining rows after removing duplicates: 14989


### 🔧 **Try It Yourself - Part 4**

4.1. Use `copied_df.duplicated().sum()` to count how many duplicates are in your deep copy dataset.

4.2. Try using `copied_df.drop_duplicates(keep='last')` instead. earlier the default `keep='first'` was used. Understand the difference between keeping first vs last.


In [53]:
# 🔧 4.1. Add code here


# 🔧 4.2. Add code here


# 🔧 4.3. Add code here




🔧 4.3. Explore whether duplicate rows share the same ID or just values across all columns. Write any necessary code to view the duplicate records, and comment on your observation.

### ✍️ Your Response: 🔧

4.3.








## **Part 5: Identify and Remove Obvious Outliers**

Now we will go back to our regular DataFrame `df`.

Outliers are values that fall far outside the normal range. They can come from data entry mistakes or rare cases.

- Use summary statistics such as `df.describe()` or visual tools (like boxplots) to find them.
- Look for clearly unrealistic values — e.g., negative prices or extremely high data usage.
- Decide how to handle them:
  - ***remove*** if they're errors. We can apply sensible business rules.
  - ***Keep*** if they're valid but rare - or ***cap*** them if needed.

Outliers can distort averages, stretch visualizations, and mislead models, so it’s important to address them carefully.



In [54]:
# View the numeric and date variables
df.describe()

,id,income,data_overage_mb,data_leftover_mb,data_mb_used,text_message_count,house,handset_price,over_15mins_calls_per_month,average_call_duration
count,14989.000000,14989.000000,14989.000000,14989.000000,14989.000000,14989.000000,1.498900e+04,14989.000000,14986.000000,14989.000000
mean,11859.840883,241982.463940,153.526586,37.261392,4201.141370,135.951431,8.771716e+05,795.006938,10.570266,10.064647
std,6813.059020,109611.297837,113.009107,28.118539,2196.798199,62.952477,2.870281e+05,1235.894864,8.401910,41.087897
min,2.000000,-65000.000000,0.000000,0.000000,400.000000,52.000000,-4.630000e+02,-200.000000,0.000000,1.000000
25%,6137.000000,147806.000000,54.000000,12.000000,2304.000000,93.000000,6.443860e+05,499.000000,3.000000,5.000000
50%,11762.000000,241656.000000,151.000000,34.000000,4221.000000,135.000000,8.764130e+05,777.000000,9.000000,10.000000
75%,17398.000000,336443.000000,242.000000,62.000000,6063.000000,178.000000,1.098843e+06,1062.000000,17.000000,14.000000
max,25354.000000,432000.000000,380.000000,89.000000,8000.000000,5000.000000,1.456389e+06,125000.000000,35.000000,5000.000000


In [55]:
# View shape before outlier filtering
print("Shape before removing obvious outliers:", df.shape)

# Remove negative or nonsensical values using business rules

# Example: remove rows where 'handset_price' is negative
df = df[df['handset_price'] >= 0]

# Example: remove rows with unusually long call durations
df = df[df['average_call_duration'] < 1000]

# Example: remove rows with extremely high text message counts
df = df[df['text_message_count'] < 3000]

# View shape after outlier filtering
print("Shape after removing obvious outliers:", df.shape)


Shape before removing obvious outliers: (14989, 16)
Shape after removing obvious outliers: (14986, 16)


### 🔧 **Try It Yourself - Part 5**

5.1. Use `df.describe()` to look for columns with extreme minimum or maximum values.

5.2. Set a threshold each for what you think is "too high" for:
  - `data_mb_used`
  - `over_15mins_calls_per_month`

5.3. Remove those outliers using boolean filtering like `df = df[df['column'] < threshold]` and then view the size of the df after this step

In [56]:
# 🔧 5.1 add code here


# 🔧 5.2 add code here:


# 🔧 5.3 add code here:



## **Part 6: Handle Outliers Using Quantiles**

Instead of removing outliers, which would reduce our sample size, we can limit their impact by keeping the rows but capping extreme values — a method known as **Winsorizing**.

### How to Do It:
- Use `.quantile()` to identify the 1st and 99th percentiles (or other thresholds).
- Use `.clip()` to cap values to be within that range.

This keeps your dataset intact (keeps all rows) while reducing the influence of extreme values on your analysis or model.




In [57]:
# Calculate 1st and 99th percentiles for income
income_min, income_max = df['income'].quantile([0.01, 0.99])

# Use .loc to avoid SettingWithCopyWarning and ensure assignment modifies the original DataFrame
df.loc[:, 'income'] = df['income'].clip(lower=income_min, upper=income_max)

# Clip 'data_mb_used' to within 1st and 99th percentiles
usage_min, usage_max = df['data_mb_used'].quantile([0.01, 0.99])
df.loc[:, 'data_mb_used'] = df['data_mb_used'].clip(lower=usage_min, upper=usage_max)

# Clip 'average_call_duration' to reduce the effect of extreme outliers
call_min, call_max = df['average_call_duration'].quantile([0.01, 0.99])
df.loc[:, 'average_call_duration'] = df['average_call_duration'].clip(lower=call_min, upper=call_max)

# View shape after outlier filtering
print("Shape after removing obvious outliers:", df.shape)



Shape after removing obvious outliers: (14986, 16)


### 🔧 **Try It Yourself - Part 6**

6.1. Use `.quantile([0.01, 0.99])` to save the min and max thresholds for outliers in:
  - `text_message_count`
  - `over_15mins_calls_per_month`

6.2. Call `.describe()` before clipping these variables. Apply `.clip(lower=..., upper=...)` to reduce the impact of outliers in each of the above variables.




In [58]:
# 🔧 6.1 Add code here


# 🔧 6.2 Add code here


# 🔧 6.3 Add code here



🔧 6.3. Compare the `.describe()` output on these clipped variables before and after clipping and comment on what you observe. 🔧

### ✍️ Your Response: 🔧
6.3



## **Part 7. Save the modified and cleaned file**

Often after cleaning, we will want to save the newly cleaned file for future use. Use `df.to_csv('newfilename', index = False)` to save the contents without adding a new default index (0,1,2,......)



In [59]:
df.to_csv('megatelco_cleaned.csv', index=False)

The file will be added to your temporary Google Colab virtual session. After running the above command, you can view this file by clicking on the folder icon in the very left hand side menu of this page.

**Please right click and download the file to your local machine, so you can use this in the next lab.**

##  **Part 8: Reflection (100 words or less per question)**



### 🔧 **Answer overall reflection questions - Part 8**
8.1. Which step fixed the most issues in the dataset?

8.2. What surprised you about the structure or values?

8.3. Do you feel this data is now ready for transformation in Lab 7?



### ✍️ Your Responses: 🔧


🔧 8.1


🔧 8.2


🔧 8.3




## Export Your Notebook to Submit in Canvas
- Use the instructions from Lab 1

In [60]:
!jupyter nbconvert --to html "lab_06_LastnameFirstname.ipynb"

[NbConvertApp] WARNING | pattern 'lab_06_LastnameFirstname.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
